In [ ]:
import subprocess, sys

def pip(*args):
    r = subprocess.run(
        [sys.executable,"-m","pip","install","-q","--no-cache-dir",*args],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"❌ {r.stderr[-500:]}")
        raise RuntimeError("pip failed")

print("1/4 numpy + scipy...")
pip("numpy>=2.0.0,<3.0", "scipy>=1.14.0")

print("2/4 YOLO + spaCy...")
pip("ultralytics>=8.2,<9.0")
pip("spacy>=3.7,<4.0")

subprocess.run(
    [sys.executable,"-m","spacy","download","en_core_web_sm"],
    check=True
)

print("3/4 CLIP...")
pip("ftfy", "regex", "tqdm")
pip("git+https://github.com/openai/CLIP.git")
pip("open-clip-torch")

print("4/4 utilities...")
pip(
    "pandas>=2.2.0",
    "Pillow>=10.0,<13.0",
    "matplotlib>=3.8",
    "seaborn>=0.13"
)

print("\n✅ Done. → Restart session → Ctrl+F9")

1/4 numpy + scipy...
2/4 YOLO + spaCy...
3/4 CLIP...
4/4 utilities...

✅ Done. → Restart session → Ctrl+F9


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR   = Path("/content/drive/MyDrive/text_to_image")
OUTPUT_DIR = BASE_DIR / "output"
DATA_DIR = BASE_DIR / "data"

EVAL_DIR = BASE_DIR / "evaluate"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_OUT  = Path("/content/output")
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

assert OUTPUT_DIR.exists(), f"❌ output folder not found: {OUTPUT_DIR}"
imgs = list(OUTPUT_DIR.glob("*.png"))
print(f"✅ Drive mounted | Found {len(imgs)} images in output/")


Mounted at /content/drive
✅ Drive mounted | Found 50 images in output/


##IMPORTS & CONSTANTS

In [ ]:
import gc, json, datetime, shutil
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import spacy

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

YOLO_CONF        = 0.25

SKIP_NOUNS = {"picture","photo","photograph","image",
              "thing","something","way","lot","kind"}

nlp = spacy.load("en_core_web_sm")
print("✅ Imports OK")


✅ Imports OK


##LOAD PROMPTS & IMAGES

In [ ]:
csv_path = DATA_DIR / "prompts.csv"
assert csv_path.exists(), "❌ prompts.csv not found — did Part 1 finish?"

df = pd.read_csv(csv_path)
print(f"✅ {len(df)} prompts loaded")

df["image_path"] = df["prompt_id"].apply(
    lambda pid: OUTPUT_DIR / f"{int(pid):04d}.png"
)
missing = df[~df["image_path"].apply(lambda p: p.exists())]
if len(missing):
    print(f"⚠️  Missing images: {missing['prompt_id'].tolist()}")
else:
    print(f"✅ All {len(df)} images found")

def extract_nouns(text: str) -> List[str]:
    doc = nlp(text.lower())
    return [t.lemma_ for t in doc
            if t.pos_ in ("NOUN","PROPN") and t.lemma_ not in SKIP_NOUNS]

df["nouns"] = df["prompt"].apply(extract_nouns)


✅ 49 prompts loaded
✅ All 49 images found


##*CLIP evaluation*

In [ ]:
import torch
import clip
from PIL import Image
from pathlib import Path
from typing import Dict, List
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

print("✅ CLIP model ready on:", device)

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 233MiB/s]


✅ CLIP model ready on: cpu


In [ ]:
def compute_clip_score(image_path: Path, prompt: str) -> Dict:
    """
    Returns semantic similarity between image and text prompt.
    Higher = better alignment.
    """

    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    text  = clip.tokenize([prompt]).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features  = model.encode_text(text)

        image_features = F.normalize(image_features, dim=-1)
        text_features  = F.normalize(text_features, dim=-1)

        score = (image_features @ text_features.T).item()

    return {
        "clip_score": float(score)
    }

In [ ]:
print("Running CLIP evaluation...")

clip_scores = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="CLIP eval"):
    img_path = row["image_path"]
    prompt   = row["prompt"]

    if not img_path.exists():
        clip_scores.append(0.0)
        continue

    score = compute_clip_score(img_path, prompt)
    clip_scores.append(score["clip_score"])

df_clip = df[["prompt_id", "prompt", "image_path"]].copy()
df_clip["clip_score"] = clip_scores

mean_clip = df_clip["clip_score"].mean()

print(f"\n✅ Mean CLIP score: {mean_clip:.4f}")

Running CLIP evaluation...


CLIP eval:   0%|          | 0/49 [00:00<?, ?it/s]


✅ Mean CLIP score: 0.3092


In [ ]:
csv_path = EVAL_DIR / "clip_results.csv"

df_clip.to_csv(csv_path, index=False)
print("✅ Saved CSV:", csv_path)

✅ Saved CSV: /content/drive/MyDrive/text_to_image/evaluate/clip_results.csv


In [ ]:
print("\n🔻 Worst CLIP matches:")
display(df_clip.nsmallest(5, "clip_score")[["prompt_id", "clip_score", "prompt"]])

print("\n🔺 Best CLIP matches:")
display(df_clip.nlargest(5, "clip_score")[["prompt_id", "clip_score", "prompt"]])


🔻 Worst CLIP matches:


,prompt_id,clip_score,prompt
8,9,0.257257,An open laptop computer sitting on top of a wo...
21,22,0.262442,Single sheep in a field looking back at camera.
16,17,0.265504,A person putting doughnuts into a bag in a shop.
26,27,0.269667,A large combination pizza with two pieces gone.
15,16,0.271794,A professional photograph of a motorcycle ride...



🔺 Best CLIP matches:


,prompt_id,clip_score,prompt
27,28,0.382412,A bearded man in a suit eating pizza.
1,2,0.372395,A zebra chews a flower in a fenced in field.
44,45,0.359133,Tabby cat with green eyes wearing a hat
19,20,0.355391,A picture of pink bathroom sink and a mirror.
36,37,0.343656,A man is skate boarding down a path and a dog ...


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12,4))

ax.bar(df_clip["prompt_id"], df_clip["clip_score"], color="#2E86AB", edgecolor="white")
ax.axhline(mean_clip, color="black", linestyle="--", label=f"Mean={mean_clip:.3f}")

ax.set_title("CLIP Score per Prompt (Semantic Alignment)")
ax.set_xlabel("Prompt ID")
ax.set_ylabel("CLIP similarity")

plt.xticks(rotation=90, fontsize=6)
plt.legend()
plt.tight_layout()

plot_path = EVAL_DIR / "clip_scores.png"

plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print("✅ Saved plot:", plot_path)

✅ Saved plot: /content/drive/MyDrive/text_to_image/evaluate/clip_scores.png


##*YOLO evaluation*

In [ ]:
from ultralytics import YOLO
import torch
import open_clip
from pathlib import Path
import pandas as pd
import datetime
import json

yolo = YOLO("yolov8n.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model, _, clip_tokenizer = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)
clip_model = clip_model.to(device)
clip_model.eval()

print("✅ YOLO + CLIP ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


✅ YOLO + CLIP ready


In [ ]:
def detect_objects(image_path: Path, conf: float = YOLO_CONF):
    results = yolo(str(image_path), conf=conf, verbose=False)
    names = yolo.names

    return list({
        names[int(c)]
        for r in results
        for c in r.boxes.cls.tolist()
    })

In [ ]:
def encode_texts(text_list):
    tokens = open_clip.tokenize(text_list).to(device)
    with torch.no_grad():
        feats = clip_model.encode_text(tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats

In [ ]:
def semantic_match(expected, detected):

    if len(expected) == 0 or len(detected) == 0:
        return {
            "tp": 0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "expected": expected,
            "detected": detected
        }

    exp_feat = encode_texts(expected)
    det_feat = encode_texts(detected)

    sim = exp_feat @ det_feat.T

    tp = 0
    used = set()

    for i in range(sim.shape[0]):
        best = torch.argmax(sim[i]).item()

        if best not in used and sim[i][best] > 0.25:
            tp += 1
            used.add(best)

    precision = tp / len(detected) if detected else 0
    recall = tp / len(expected) if expected else 0

    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0.0)

    return {
        "tp": tp,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "expected": expected,
        "detected": detected
    }

In [ ]:
print("Running YOLO + CLIP evaluation...")

rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="eval"):

    img_path = row["image_path"]
    expected = row["nouns"]

    if not img_path.exists():
        rows.append({
            "prompt_id": row["prompt_id"],
            "prompt": row["prompt"],
            "f1": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "expected": [],
            "detected": []
        })
        continue

    detected = detect_objects(img_path)
    sc = semantic_match(expected, detected)

    rows.append({
        "prompt_id": row["prompt_id"],
        "prompt": row["prompt"],
        "f1": sc["f1"],
        "precision": sc["precision"],
        "recall": sc["recall"],
        "expected": sc["expected"],
        "detected": sc["detected"]
    })

eval_df = pd.DataFrame(rows)

mean_f1 = eval_df["f1"].mean()
mean_precision = eval_df["precision"].mean()
mean_recall = eval_df["recall"].mean()

print("\n✅ Mean F1:", mean_f1)
print("✅ Mean Precision:", mean_precision)
print("✅ Mean Recall:", mean_recall)

Running YOLO + CLIP evaluation...


eval:   0%|          | 0/49 [00:00<?, ?it/s]


✅ Mean F1: 0.5303045027534823
✅ Mean Precision: 0.7816812439261418
✅ Mean Recall: 0.4495626822157434


In [ ]:
eval_path = EVAL_DIR / "yolo_clip_results.csv"

eval_df.to_csv(eval_path, index=False)

print("✅ Saved:", eval_path)

✅ Saved: /content/drive/MyDrive/text_to_image/evaluate/yolo_clip_results.csv


In [ ]:
print("\n🔻 Worst YOLO matches:")
display(eval_df.nsmallest(5, "f1")[[
    "prompt_id", "f1", "precision", "recall", "prompt"
]])

print("\n🔺 Best YOLO matches:")
display(eval_df.nlargest(5, "f1")[[
    "prompt_id", "f1", "precision", "recall", "prompt"
]])


🔻 Worst YOLO matches:


,prompt_id,f1,precision,recall,prompt
17,18,0.000000,0.000000,0.000000,A very ornately decorated and brightly colored...
22,23,0.285714,0.500000,0.200000,A few pizza slices next to a couple of bread s...
32,33,0.285714,0.333333,0.250000,A dim room with toilet bowls lined along the wall
19,20,0.333333,0.333333,0.333333,A picture of pink bathroom sink and a mirror.
34,35,0.333333,0.333333,0.333333,A slightly made/messy bed against the corner i...



🔺 Best YOLO matches:


,prompt_id,f1,precision,recall,prompt
29,30,0.857143,1.0,0.750000,People flying kites in a park next to a lake.
43,44,0.857143,1.0,0.750000,a woman on a tennis court getting ready to ser...
9,10,0.800000,1.0,0.666667,A woman twirling an umbrella with flowers on it.
11,12,0.800000,1.0,0.666667,A woman holding a purse and a cellphone.
15,16,0.800000,1.0,0.666667,A professional photograph of a motorcycle ride...


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12,4))

ax.bar(eval_df["prompt_id"], eval_df["f1"], color="#E84855", edgecolor="white")
ax.axhline(mean_f1, color="black", linestyle="--", label=f"Mean F1={mean_f1:.3f}")

ax.set_title("Semantic YOLO F1 (CLIP-aligned)")
ax.set_xlabel("Prompt ID")
ax.set_ylabel("F1 score")

plt.xticks(rotation=90, fontsize=6)
plt.legend()
plt.tight_layout()

plot_path = EVAL_DIR / "semantic_f1.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print("✅ Saved plot:", plot_path)

✅ Saved plot: /content/drive/MyDrive/text_to_image/evaluate/semantic_f1.png


##Final score

In [ ]:
alpha = 0.5

eval_df["clip_score"] = df_clip["clip_score"]

eval_df["final_score"] = (
    alpha * eval_df["clip_score"] +
    (1 - alpha) * eval_df["f1"]
)

In [ ]:
print("\n🔻 Worst FINAL matches:")
display(eval_df.nsmallest(5, "final_score")[[
    "prompt_id", "final_score", "f1", "clip_score", "prompt"
]])

print("\n🔺 Best FINAL matches:")
display(eval_df.nlargest(5, "final_score")[[
    "prompt_id", "final_score", "f1", "clip_score", "prompt"
]])


🔻 Worst FINAL matches:


,prompt_id,final_score,f1,clip_score,prompt
17,18,0.164345,0.000000,0.328690,A very ornately decorated and brightly colored...
22,23,0.288524,0.285714,0.291334,A few pizza slices next to a couple of bread s...
32,33,0.291825,0.285714,0.297936,A dim room with toilet bowls lined along the wall
14,15,0.302967,0.333333,0.272600,Night picture of a car parked and some parking...
34,35,0.313089,0.333333,0.292844,A slightly made/messy bed against the corner i...



🔺 Best FINAL matches:


,prompt_id,final_score,f1,clip_score,prompt
29,30,0.592571,0.857143,0.327998,People flying kites in a park next to a lake.
43,44,0.583650,0.857143,0.310157,a woman on a tennis court getting ready to ser...
37,38,0.569320,0.800000,0.338639,A man on a long exposure picture riding an ele...
27,28,0.566206,0.750000,0.382412,A bearded man in a suit eating pizza.
35,36,0.554410,0.800000,0.308819,A bench sit in front of a blue and yellow train.


In [ ]:
mean_final = eval_df["final_score"].mean()

print("\n✅ Mean FINAL score:", round(mean_final, 4))


✅ Mean FINAL score: 0.4198


In [ ]:
final_df = eval_df[[
    "prompt_id",
    "prompt",
    "f1",
    "clip_score",
    "final_score"
]].copy()

final_path = EVAL_DIR / "final_results.csv"
final_df.to_csv(final_path, index=False)

print("✅ Saved:", final_path)

✅ Saved: /content/drive/MyDrive/text_to_image/evaluate/final_results.csv
